# AkidaNet on ImageNet - Evaluation

<p align="right">Run Time: ~5 minutes (sample pack) / ~25 minutes (full validation set)</p>

This notebook works through the published **AkidaNet** ImageNet models: what makes the
architecture Akida-compatible, how the full-precision, quantized and converted models
compare, and how to reuse the weights as a backbone for your own task.

There is no training here. AkidaNet takes days to train on ImageNet, so this example
starts from the published weights. If you want to see a full train-quantize-convert
pipeline, the [plant_village](../plant_village) example runs one in about 20 minutes.

The notebook covers:

1. The three architectural changes that separate AkidaNet from MobileNet v1
2. Evaluation of the float, quantized (QAT) and Akida models
3. Activation sparsity - why the model is cheap to run on Akida
4. Extracting a backbone for transfer learning

In [ ]:
# Colab-only setup. Local users: ignore this cell - it does nothing for you.
import sys
if 'google.colab' in sys.modules:
    !wget -q https://raw.githubusercontent.com/Brainchip-Inc/brainchip_devhub/main/akida1/model_zoo/imagenet_akidanet/colab_setup.py
import colab_setup
colab_setup.setup()

## Configuration

Pick one model by width multiplier (`ALPHA`) and input resolution (`RESOLUTION`).

Accuracy is measured on the full 50,000-image ImageNet validation set if you have it set
up. ImageNet cannot be redistributed, so that setup is manual - see 'Dataset setup' in the
README. If the dataset is not found, the notebook falls back to a 10-image sample pack that
is fetched automatically. That fallback is enough to exercise the whole pipeline, but far
too small to measure accuracy with, and the notebook will say so rather than pretend
otherwise.

In [ ]:
import os
import numpy as np

ALPHA = 1.0          # 0.25, 0.5 or 1.0
RESOLUTION = 224     # 160 or 224
BATCH_SIZE = 128

DATA_PATH = './data/imagenet_tfds'
INPUT_SHAPE = (RESOLUTION, RESOLUTION, 3)

# Is the full validation set available?
USE_FULL_VALIDATION = os.path.isdir(os.path.join(DATA_PATH, 'tfds', 'data'))
print(f'Model: alpha={ALPHA}, {RESOLUTION}x{RESOLUTION}')
print('Evaluating on:', 'full ImageNet validation set (50,000 images)'
      if USE_FULL_VALIDATION else '10-image sample pack (smoke test only)')

## The AkidaNet architecture

AkidaNet is **MobileNet v1** reshaped to map cleanly onto Akida 1. It keeps
depthwise-separable convolutions as the main efficiency lever, and changes three things.

**1. The first four blocks are standard convolutions.** MobileNet uses separable
convolutions all the way down. AkidaNet uses full convolutions for `conv_0` through
`conv_3`, switching to separable from `separable_4` on. These early layers are narrow (32
to 128 filters at alpha 1.0), so a standard convolution there costs little in absolute
terms while adding real expressivity where the low-level features are built.

**2. No ReLU between the depthwise and pointwise stages.** The Akida 1 separable
convolution is a *fused* primitive - the two stages execute as one operation, leaving
nowhere to put an activation between them. So the block is depthwise, pointwise, BN, ReLU.

**3. Global average pooling comes before the final ReLU.** The Akida 1 GAP implementation
requires it to sit before the neighbouring ReLU, so the last block ends convolution,
pooling, BN, ReLU rather than the usual convolution, BN, ReLU, pooling. ReLU is monotonic
but not linear, so this genuinely changes the computation - the models are trained with it.
The cell below prints the layer order so you can check this rather than take it on trust.

Points 2 and 3 are selected automatically by `akida_models` when the Akida v1 context is
active. This matters: the **default context is v2**, which builds the unfused,
post-ReLU-pooling variant and will not match these weights. Hence
`with set_akida_version(AkidaVersion.v1):` around any model construction.

In [ ]:
from cnn2snn import load_quantized_model
from imagenet_akidanet_model import model_path

model = load_quantized_model(str(model_path(ALPHA, RESOLUTION, 'float')))

# The 224 checkpoints carry a stale internal name saying '160' - they were produced by
# rescaling the 160 models. Always read the resolution from input_shape, never from name.
print(f'model.name       = {model.name}   <- do not trust this')
print(f'model.input_shape = {model.input_shape}   <- trust this')
print(f'parameters        = {model.count_params():,}')

### Seeing the three changes in the built model

Rather than take the description on trust, inspect the layers.

In [ ]:
from tf_keras.layers import Conv2D, SeparableConv2D, GlobalAveragePooling2D, ReLU

# 1. Standard convolutions first, separable afterwards
print('Convolution types in order:')
for layer in model.layers:
    if isinstance(layer, (Conv2D, SeparableConv2D)) and 'classifier' not in layer.name:
        kind = 'separable' if isinstance(layer, SeparableConv2D) else 'standard '
        print(f'  {kind}  {layer.name:<24} filters={layer.filters}')

In [ ]:
# 2. Fused separable convolution: a single SeparableConv2D layer, so there is no
#    intermediate activation between the depthwise and pointwise stages.
sep = [l for l in model.layers if isinstance(l, SeparableConv2D)][0]
print(f'{sep.name}: one fused layer of type {type(sep).__name__}')
print('   depthwise kernel:', sep.depthwise_kernel.shape)
print('   pointwise kernel:', sep.pointwise_kernel.shape)
print('   -> no ReLU can be applied between them\n')

# 3. Global average pooling sits BEFORE the neighbouring ReLU
names = [l.name for l in model.layers]
gap = [l for l in model.layers if isinstance(l, GlobalAveragePooling2D)][0]
idx = names.index(gap.name)
print('Layer order around the global average pooling:')
for l in model.layers[idx - 1: idx + 3]:
    print(f'   {type(l).__name__:<24} {l.name}')

## Data

The preprocessing is the standard ImageNet validation recipe: aspect-preserving resize so
the shorter side becomes `round(size * 1.143)`, then a centre crop.

Two things are deliberately absent, and both catch people out: there is **no mean
subtraction** and **no division by 255**. The model contains a `Rescaling` layer that
computes `x / 128 - 1` internally, so the pipeline hands it plain uint8 pixels. Normalising
the data yourself is the most common cause of a pretrained model scoring 0.1%.

In [ ]:
from imagenet_akidanet_data import get_data, get_labelled_samples, index_to_label

if USE_FULL_VALIDATION:
    dataset, num_examples = get_data(DATA_PATH, INPUT_SHAPE, BATCH_SIZE)
    print(f'Validation set: {num_examples:,} images')
    images, labels = get_labelled_samples(INPUT_SHAPE)   # still handy for display
else:
    images, labels = get_labelled_samples(INPUT_SHAPE)
    print(f'Sample pack: {len(images)} images, dtype={images.dtype}, '
          f'range [{images.min()}, {images.max()}]')

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 5, figsize=(15, 6.5))
for ax, img, label in zip(axes.ravel(), images, labels):
    ax.imshow(img)
    ax.set_title(index_to_label(label).split(',')[0], fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## Float evaluation

The full-precision model, as a baseline for everything that follows.

In [ ]:
from imagenet_akidanet_eval import evaluate_keras_model, run_smoke_test

if USE_FULL_VALIDATION:
    float_t1, float_t5 = evaluate_keras_model(model, dataset)
else:
    float_t1, float_t5 = run_smoke_test(model, False, INPUT_SHAPE)

print(f'\nFloat: top-1 {float_t1 * 100:.2f}%, top-5 {float_t5 * 100:.2f}%')

## Quantized model (QAT)

Akida 1 runs 4-bit weights and activations, with an 8-bit input layer. Quantizing to those
widths costs some accuracy, and quantization-aware training recovers most of it by
fine-tuning the model with the quantizers in place.

These models were quantized and fine-tuned upstream, so we load the result directly. The
`plant_village` example shows the `cnn2snn quantize` and QAT steps being run.

In [ ]:
qat_model = load_quantized_model(str(model_path(ALPHA, RESOLUTION, 'qat')))

if USE_FULL_VALIDATION:
    qat_t1, qat_t5 = evaluate_keras_model(qat_model, dataset)
else:
    qat_t1, qat_t5 = run_smoke_test(qat_model, False, INPUT_SHAPE)

print(f'\nQAT: top-1 {qat_t1 * 100:.2f}%, top-5 {qat_t5 * 100:.2f}%')
print(f'Change vs float: {(qat_t1 - float_t1) * 100:+.2f} points top-1')

## Akida model

`cnn2snn convert` turns the quantized Keras model into an Akida model. If no hardware is
present it runs on the software backend, which is what happens on Colab.

The converted model should score essentially the same as the QAT model - conversion is a
change of representation, not of the computation.

In [ ]:
import akida
from imagenet_akidanet_eval import evaluate_akida_model

akida_model = akida.Model(str(model_path(ALPHA, RESOLUTION, 'akida')))
akida_model.summary()

In [ ]:
if USE_FULL_VALIDATION:
    akida_t1, akida_t5, n = evaluate_akida_model(akida_model, dataset)
else:
    akida_t1, akida_t5 = run_smoke_test(akida_model, True, INPUT_SHAPE)

print(f'\nAkida: top-1 {akida_t1 * 100:.2f}%, top-5 {akida_t5 * 100:.2f}%')

## Activation sparsity

This is the number that explains why the model is cheap to run on Akida. Akida is
event-driven: a zero activation generates no event and therefore no work. The higher the
sparsity, the less the hardware actually does.

It is also why the hardware benchmark must use real images. Feed random noise and the
activity statistics are wrong, so the latency and power figures are wrong too.

In [ ]:
from akida_models.sparsity import compute_sparsity
from brainchip_utils.plot_utils import pretty_print_sparsity
from imagenet_akidanet_data import get_samples

samples = get_samples(INPUT_SHAPE, num_samples=100,
                      data_path=DATA_PATH if USE_FULL_VALIDATION else None)
sparsity_dict = compute_sparsity(akida_model, samples=samples)
pretty_print_sparsity(sparsity_dict)
print(f'\nMean activation sparsity: {np.mean(list(sparsity_dict.values())) * 100:.2f}%')

## Using the model as a transfer-learning backbone

This is the other reason these weights are in the repository. Training a good image model
from scratch needs a large dataset; fine-tuning an ImageNet-pretrained backbone needs a
small one and a few minutes.

`load_akidanet_backbone()` gives you the network without its 1000-class head, weights
loaded, in the Akida v1 context. Attach your own head and fine-tune.

Note that `akida_models.akidanet_imagenet_pretrained()` does something similar, but only
knows about the **224** checkpoints - the 160 weights are reachable only by loading them
directly, which is what this helper does for both resolutions.

In [ ]:
from imagenet_akidanet_model import load_akidanet_backbone

backbone = load_akidanet_backbone(alpha=ALPHA, resolution=RESOLUTION)
print(f'Backbone output shape: {backbone.output_shape}')
print(f'Backbone parameters:   {backbone.count_params():,}')
print(f'Full model parameters: {model.count_params():,}  '
      f'(the difference is the 1000-class classifier)')

In [ ]:
from akida_models.layer_blocks import dense_block
from cnn2snn import set_akida_version, AkidaVersion
from tf_keras import Model

NUM_CLASSES = 10   # your task

# Build the new head inside the v1 context, so its activations stay Akida 1 compatible
with set_akida_version(AkidaVersion.v1):
    x = dense_block(backbone.output, units=NUM_CLASSES, name='predictions',
                    add_batchnorm=False, relu_activation=False)
    my_model = Model(backbone.input, x, name='akidanet_my_task')

print(f'Ready to fine-tune: {my_model.input_shape} -> {my_model.output_shape}')
print('\nRemember: feed uint8 pixels. The rescaling layer is inside the model.')

See [plant_village_model.py](../plant_village/plant_village_model.py) for a complete worked
version of this, fine-tuning the alpha 0.5 backbone to 38 classes and reaching over 99%
accuracy in about 20 minutes.

## Summary

In [ ]:
print(f'AkidaNet alpha={ALPHA}, {RESOLUTION}x{RESOLUTION}')
print(f'  measured on: {"50,000 validation images" if USE_FULL_VALIDATION else "10 sample images (smoke test)"}')
print()
print(f'  {"":8}{"top-1":>10}{"top-5":>10}')
print(f'  {"float":8}{float_t1 * 100:9.2f}%{float_t5 * 100:9.2f}%')
print(f'  {"QAT":8}{qat_t1 * 100:9.2f}%{qat_t5 * 100:9.2f}%')
print(f'  {"Akida":8}{akida_t1 * 100:9.2f}%{akida_t5 * 100:9.2f}%')
print()
print(f'  mean activation sparsity: {np.mean(list(sparsity_dict.values())) * 100:.2f}%')

if not USE_FULL_VALIDATION:
    print('\nNOTE: these are 10-image smoke-test numbers, not accuracy measurements.')
    print('See the README Model Card for figures over the full validation set.')

To measure hardware latency and power on an AKD1500, continue with
[imagenet_akidanet_notebook_benchmark.ipynb](imagenet_akidanet_notebook_benchmark.ipynb).